# Module 2 Homework (2026 Cohort)

Solutions for the Module 2 homework of the Stock Markets Analytics Zoomcamp:

* **Q1–Q3:** IPO data from [iposcoop.com](https://www.iposcoop.com/) combined with daily prices from Yahoo Finance
* **Q4:** a simple RSI strategy on the course dataset with precomputed indicators
* **Q5:** a few numbers for the open question

The code follows the livestream snippets. Each question has its own section, and the answer is printed at the end of it.

# 0) Imports

In [1]:
# !pip install yfinance gdown lxml pyarrow

# IMPORTS
import numpy as np
import pandas as pd
import requests
from io import StringIO

import os
import time

# hide the numpy warning that describe() gives when a Sharpe ratio is infinite (Question 2)
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

#Fin Data Sources
import yfinance as yf

# download files from Google Drive
import gdown

# 1) Question 1: Withdrawn IPOs by Company Type

* load the "Recently Filed" IPO list from https://www.iposcoop.com/ipos-recently-filed/
* keep the withdrawn IPOs filed before Sep 11, 2026
* classify the companies by name and sum the value of the withdrawn shares by company type

In [2]:
# Same request as in the livestream snippet (get_ipos_by_year): a browser User-Agent,
# and the HTML text wrapped in StringIO for read_html
url = "https://www.iposcoop.com/ipos-recently-filed/"
headers = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/58.0.3029.110 Safari/537.3'
    )
}

response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()

ipos_filed_df = pd.read_html(StringIO(response.text))[0]
ipos_filed_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   File Date             500 non-null    str    
 1   Company               500 non-null    str    
 2   Symbol                500 non-null    str    
 3   Managers              500 non-null    str    
 4   Shares (millions)     500 non-null    float64
 5   Price Low             464 non-null    str    
 6   Price High            464 non-null    str    
 7   Est $ Vol (millions)  495 non-null    str    
 8   Expected To Trade     500 non-null    str    
 9   SCOOP Rating          500 non-null    str    
dtypes: float64(1), str(9)
memory usage: 92.5 KB


In [3]:
ipos_filed_df.head()

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-09-28,Encore Medical Inc.,EMI,The Oak Ridge Financial Services Group/Network...,3.0,$5.00,$5.00,$15.00,Wednesday,S/O
1,2026-09-25,Elevation Acquisition Group,ELEVU,Maxim Group,10.0,$10.00,$10.00,$100.00,TBA,S/O
2,2026-09-25,Holtec Nuclear (Withdrawn),HNUC,J.P.Morgan/Guggenheim Securities/Goldman Sachs...,50.0,$15.00,$18.00,$825.00,Withdrawn,S/O
3,2026-09-25,RegenLab USA,RGNA,StoneX,0.0,NaN,NaN,$35.00,TBA,S/O
4,2026-09-25,TCGX Acquisition Corp. II (Stock-Only),TCXB,Guggenheim Securities,10.0,$10.00,$10.00,$100.00,TBA,S/O


In [4]:
# keep only the withdrawn IPOs
withdrawn_df = ipos_filed_df[ipos_filed_df['Expected To Trade'] == 'Withdrawn'].copy()
print(f'Withdrawn IPOs on the page: {len(withdrawn_df)}')

# the question is about the withdrawals before Sep 11, 2026 -> filter on the filing date
withdrawn_df['File Date'] = pd.to_datetime(withdrawn_df['File Date'])
withdrawn_df = withdrawn_df[withdrawn_df['File Date'] < '2026-09-11']
print(f'Withdrawn IPOs filed before 2026-09-11: {len(withdrawn_df)}')

Withdrawn IPOs on the page: 35
Withdrawn IPOs filed before 2026-09-11: 31


The page shows only the latest 500 filings, so the list changes a bit every day. The homework text mentions 32 entries; a count that is off by one or two is expected.

In [5]:
# Order matters: the first matching rule gives the Company Type
def get_company_type(company_name:str):
  if 'Technologies' in company_name:
    return 'Technologies'
  elif 'Acquisition Corp' in company_name or 'Acquisition Corporation' in company_name or 'Corp' in company_name:
    return 'Acquisition Corp'
  elif 'Inc' in company_name or 'Incorporated' in company_name:
    return 'Inc.'
  elif 'Group' in company_name:
    return 'Group'
  elif 'Ltd' in company_name or 'Limited' in company_name:
    return 'Limited'
  elif 'Holdings' in company_name or 'Holding' in company_name:
    return 'Holdings'
  else:
    return 'Other'

# check the two examples from the homework text
print(get_company_type('EUPEC International Group Ltd.'))          # Group
print(get_company_type('Xinxu Copper Industry Technology Ltd.'))   # Limited

Group
Limited


In [6]:
withdrawn_df['Company Type'] = withdrawn_df['Company'].apply(get_company_type)
withdrawn_df['Company Type'].value_counts()

Company Type
Other               8
Limited             7
Acquisition Corp    5
Holdings            4
Technologies        3
Group               3
Inc.                1
Name: count, dtype: int64

In [7]:
# '$8.00' -> 8.0 ; '-' or missing -> None
def get_price(price):
  try:
    return float(str(price).replace('$', '').replace(',', ''))
  except ValueError:
    return None

withdrawn_df['Avg_price'] = (withdrawn_df['Price Low'].apply(get_price) + withdrawn_df['Price High'].apply(get_price)) / 2

In [8]:
# remove '$' and ',' and convert to numbers (defensive errors='coerce', as in the livestream)
withdrawn_df['Shares (millions)'] = pd.to_numeric(withdrawn_df['Shares (millions)'].astype(str).str.replace(',', ''), errors='coerce')
withdrawn_df['Est $ Vol (millions)'] = pd.to_numeric(withdrawn_df['Est $ Vol (millions)'].astype(str).str.replace('$', '').str.replace(',', ''), errors='coerce')

In [9]:
# Shares x Avg_price, and when it is missing -> Est $ Vol
withdrawn_df['Shares_offered_value'] = np.where(
    (withdrawn_df['Shares (millions)'] * withdrawn_df['Avg_price']).notnull(),
    withdrawn_df['Shares (millions)'] * withdrawn_df['Avg_price'],
    withdrawn_df['Est $ Vol (millions)'])

COLUMNS = ['File Date', 'Company', 'Company Type', 'Shares (millions)', 'Avg_price', 'Est $ Vol (millions)', 'Shares_offered_value']
withdrawn_df[COLUMNS].sort_values('Shares_offered_value', ascending=False)

,File Date,Company,Company Type,Shares (millions),Avg_price,Est $ Vol (millions),Shares_offered_value
400,2026-02-19,Clear Street Group Inc. (Withdrawn),Inc.,13.00,27.00,351.00,351.0000
190,2026-06-24,Safepoint Holdings (Withdrawn),Holdings,16.67,16.00,266.72,266.7200
65,2026-08-27,Timber Road Acquisition Corp. (Withdrawn),Acquisition Corp,20.00,10.00,200.00,200.0000
75,2026-08-24,pan-Africa Corp. (Withdrawn),Acquisition Corp,12.50,10.00,125.00,125.0000
46,2026-09-10,Motive Technologies (Withdrawn),Technologies,0.00,NaN,100.00,100.0000
169,2026-07-08,INFINT Acquisition Corp. 2 (Withdrawn),Acquisition Corp,10.00,10.00,100.00,100.0000
56,2026-09-01,Hornbeck Offshore Services (Withdrawn),Other,0.00,NaN,100.00,100.0000
189,2026-06-24,Kilcoy Global Foods Ltd. (Withdrawn),Limited,0.00,NaN,100.00,100.0000
376,2026-03-06,Invea Therapeutics (Withdrawn),Other,0.00,NaN,75.00,75.0000
230,2026-06-01,Envoy Technologies (Withdrawn – Direct Listing),Technologies,7.80,8.00,62.40,62.4000


In [10]:
withdrawn_by_type = withdrawn_df.groupby('Company Type')['Shares_offered_value'].agg(['count', 'sum']).sort_values('sum', ascending=False)
withdrawn_by_type

,count,sum
Company Type,,
Acquisition Corp,5,499.9850
Inc.,1,351.0000
Holdings,4,311.6575
Other,8,290.4450
Limited,7,197.2500
Technologies,3,184.9000
Group,3,32.5000


In [11]:
top_type = withdrawn_by_type.index[0]
top_value = withdrawn_by_type['sum'].iloc[0]
print(f'Q1 answer: {top_type} with ${top_value:.2f} million of withdrawn IPOs')

Q1 answer: Acquisition Corp with $499.99 million of withdrawn IPOs


# 2) Question 2: Median Sharpe Ratio for 2025 IPOs (First 8 Months)

* load the 2025 IPOs from https://www.iposcoop.com/2025-pricings/
* keep the IPOs offered before Sep 1, 2025 with a non-zero return
* download the daily prices with `yfinance` (same loop as in the livestream) and calculate `growth_252d`, `volatility` and `Sharpe` for every stock
* look at the stats on the trading day 2026-09-11

In [12]:
url = "https://www.iposcoop.com/2025-pricings/"
response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()

ipos_2025_df = pd.read_html(StringIO(response.text))[0]
ipos_2025_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 231 entries, 0 to 230
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Company            231 non-null    str    
 1   Symbol             231 non-null    str    
 2   Industry           231 non-null    str    
 3   Offer Date         231 non-null    str    
 4   Shares (millions)  231 non-null    float64
 5   Offer Price        231 non-null    str    
 6   1st Day Close      231 non-null    str    
 7   Current Price      231 non-null    str    
 8   Return             231 non-null    str    
 9   SCOOP Rating       231 non-null    str    
dtypes: float64(1), str(9)
memory usage: 34.4 KB


In [13]:
ipos_2025_df.head()

,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating
0,Vine Hill Capital Investment Corp. II,VHCPU,Blank Check,12/18/2025,20.0,$10.00,$0.00,$0.00,0.00%,S/O
1,Andersen Group,ANDG,Consumer Services,12/17/2025,11.0,$16.00,$23.50,$23.79,48.69%,S/O
2,Medline Inc.,MDLN,Health Care,12/17/2025,216.0,$29.00,$41.00,$43.83,51.14%,S/O
3,Wealthfront Corp.,WLTH,Financials,12/12/2025,34.6,$14.00,$14.19,$8.49,-39.36%,S/O
4,Lumexa Imaging Holdings,LMRI,Health Care,12/11/2025,25.0,$18.50,$18.52,$14.80,-20.00%,S/O


In [14]:
# convert to proper types
ipos_2025_df['Offer Date'] = pd.to_datetime(ipos_2025_df['Offer Date'], format='%m/%d/%Y')
ipos_2025_df['Return'] = pd.to_numeric(ipos_2025_df['Return'].str.replace('%', ''), errors='coerce') / 100

# IPOs before Sep 1, 2025 and with a non-zero return
filtered_ipos_df = ipos_2025_df[(ipos_2025_df['Offer Date'] < '2025-09-01') & (ipos_2025_df['Return'] != 0)]
print(f'IPOs in 2025: {len(ipos_2025_df)}')
print(f'IPOs before 2025-09-01 with a non-zero return: {len(filtered_ipos_df)}')

IPOs in 2025: 231
IPOs before 2025-09-01 with a non-zero return: 146


In [15]:
TICKERS = filtered_ipos_df['Symbol'].to_list()
len(TICKERS)

146

In [16]:
stocks_df = pd.DataFrame({'A' : []})
not_found = []

for i,ticker in enumerate(TICKERS):
  print(i,ticker)

  # Work with stock prices
  ticker_obj = yf.Ticker(ticker)
  historyPrices = ticker_obj.history(
                     period = "max",
                     interval = "1d")

  # sleep 1 sec between downloads - not to overload the API server
  time.sleep(1)

  # no data on Yahoo Finance (probably delisted) -> skip the ticker
  if historyPrices.empty:
    not_found.append(ticker)
    continue

  historyPrices['Ticker'] = ticker
  historyPrices['Date'] = historyPrices.index.date

  # growth after ~1 year of trading (252 trading days)
  historyPrices['growth_252d'] = historyPrices['Close'] / historyPrices['Close'].shift(252)

  # annualized volatility: the formula given in the homework
  historyPrices['volatility'] = historyPrices['Close'].rolling(30).std() * np.sqrt(252)

  # Sharpe ratio with a risk-free rate of 5%
  historyPrices['Sharpe'] = (historyPrices['growth_252d'] - 0.05) / historyPrices['volatility']

  if stocks_df.empty:
    stocks_df = historyPrices
  else:
    stocks_df = pd.concat([stocks_df, historyPrices], ignore_index=True)

0 PMI


1 TGHL


2 THH


3 CURX


4 CEPF


...

141 MIMI


142 MASK


143 CEPO


144 ZYBT


145 INLF


In [17]:
stocks_df['Date'] = pd.to_datetime(stocks_df['Date'])

print(f'Stocks with data on Yahoo Finance: {stocks_df.Ticker.nunique()}')
print(f'Not found on Yahoo Finance (probably delisted): {len(not_found)} {not_found}')

Stocks with data on Yahoo Finance: 132
Not found on Yahoo Finance (probably delisted): 14 ['MJID', 'EMPG', 'CAEP', 'PTNM', 'AHL', 'CEPT', 'SDM', 'TBH', 'WGRX', 'AGH', 'EPWK', 'MTSR', 'SKBL', 'MCTR']


In [18]:
# keep only the trading day 2026-09-11
sharpe_df = stocks_df[stocks_df.Date == '2026-09-11']
sharpe_df[['growth_252d', 'volatility', 'Sharpe']].describe()

,growth_252d,volatility,Sharpe
count,130.000000,131.000000,130.000000
mean,1.057613,27.516340,inf
std,3.049007,66.879183,NaN
min,0.001005,0.000000,-0.040147
25%,0.141035,2.301309,0.012042
50%,0.594534,7.885748,0.050142
75%,1.038821,23.452319,0.133619
max,33.638270,627.698106,inf


A stock with the same close price for 30 days has `volatility = 0`, so its Sharpe ratio is infinite (`inf`). This is why the mean Sharpe is `inf`, but the median is not affected.

In [19]:
sharpe_df[np.isinf(sharpe_df.Sharpe)][['Ticker', 'Close', 'growth_252d', 'volatility', 'Sharpe']]

,Ticker,Close,growth_252d,volatility,Sharpe
13474,EFTY,15.02,2.106592,0.0,inf
23654,MAMK,13.00,6.532663,0.0,inf


In [20]:
print(f"Q2 answer: median Sharpe ratio on 2026-09-11 = {sharpe_df['Sharpe'].median():.4f}")

Q2 answer: median Sharpe ratio on 2026-09-11 = 0.0501


## Expected observations

In [21]:
# 1) median vs mean growth
print(f"growth_252d median: {sharpe_df['growth_252d'].median():.4f}")
print(f"growth_252d mean:   {sharpe_df['growth_252d'].mean():.4f}")

# 2) stocks that reached 252 trading days (they have a growth_252d value)
print(f"Stocks with 252 trading days: {sharpe_df['growth_252d'].notnull().sum()} of {len(sharpe_df)}")

growth_252d median: 0.5945
growth_252d mean:   1.0576
Stocks with 252 trading days: 130 of 132


In [22]:
# 3) top-10 stocks by Sharpe ratio, and their rank by pure growth
obs_df = sharpe_df[~np.isinf(sharpe_df.Sharpe)][['Ticker', 'Close', 'growth_252d', 'volatility', 'Sharpe']].dropna()
obs_df['growth_rank'] = obs_df['growth_252d'].rank(ascending=False)
obs_df['sharpe_rank'] = obs_df['Sharpe'].rank(ascending=False)
obs_df.sort_values('sharpe_rank').head(10).round(3)

,Ticker,Close,growth_252d,volatility,Sharpe,growth_rank,sharpe_rank
7313,MAGH,6.760,5.452,0.000,4204751.250,3.0,1.0
14911,HCMAU,10.470,1.035,0.183,5.385,32.0,2.0
1355,CEPF,10.355,1.024,0.352,2.771,34.0,3.0
32301,TMDE,0.749,0.776,0.415,1.751,52.0,4.0
23345,TLIH,5.750,0.990,0.649,1.449,37.0,5.0
67322,CEPO,10.755,1.031,0.795,1.234,33.0,6.0
33392,HXHX,0.471,0.410,0.477,0.753,82.0,7.0
59728,FBGL,0.375,0.577,0.838,0.629,65.0,8.0
1631,ETS,1.500,1.559,2.868,0.526,12.0,9.0
14620,CTW,2.340,1.164,2.590,0.430,26.0,10.0


# 3) Question 3: 'Fixed Months Holding Strategy'

Buy every stock from Question 2 at the close of its first trading day and hold it for 1 to 12 months (1 month = 21 trading days). Which holding period gives the highest median growth?

In [23]:
# future growth for 1..12 months, calculated per ticker (groupby), so the shift doesn't mix different stocks
for m in range(1, 13):
  stocks_df['future_growth_'+str(m)+'_m'] = stocks_df.groupby('Ticker')['Close'].shift(-21*m) / stocks_df['Close']

FUTURE_GROWTH = [k for k in stocks_df.keys() if k.startswith('future_growth')]
FUTURE_GROWTH

['future_growth_1_m',
 'future_growth_2_m',
 'future_growth_3_m',
 'future_growth_4_m',
 'future_growth_5_m',
 'future_growth_6_m',
 'future_growth_7_m',
 'future_growth_8_m',
 'future_growth_9_m',
 'future_growth_10_m',
 'future_growth_11_m',
 'future_growth_12_m']

In [24]:
# first trading day of every ticker
min_dates_df = stocks_df.groupby('Ticker')['Date'].min().reset_index()
min_dates_df.columns = ['Ticker', 'min_date']
min_dates_df.head()

,Ticker,min_date
0,AAPG,2025-01-24
1,AARD,2025-02-13
2,ADVB,2025-03-06
3,AII,2025-05-08
4,AIRO,2025-06-13


In [25]:
# inner join: keep only the first trading day of every stock
first_day_df = pd.merge(min_dates_df, stocks_df, left_on=['Ticker', 'min_date'], right_on=['Ticker', 'Date'], how='inner')
print(f'Stocks: {len(first_day_df)}')
first_day_df[['Ticker', 'Date', 'Close'] + FUTURE_GROWTH].head()

Stocks: 132


,Ticker,Date,Close,future_growth_1_m,future_growth_2_m,future_growth_3_m,future_growth_4_m,future_growth_5_m,future_growth_6_m,future_growth_7_m,future_growth_8_m,future_growth_9_m,future_growth_10_m,future_growth_11_m,future_growth_12_m
0,AAPG,2025-01-24,17.379999,1.159954,1.056962,1.512946,1.485328,2.348101,2.259494,2.459436,2.176640,2.040276,1.842923,1.604143,1.472382
1,AARD,2025-02-13,14.310000,0.647100,0.555556,0.635220,0.758211,0.835779,0.753319,0.716981,0.973445,0.758910,1.011181,1.072676,0.870021
2,ADVB,2025-03-06,73.000000,0.920548,0.523288,0.255069,0.181370,0.099178,0.114521,0.132329,0.124932,0.104110,0.095890,0.065479,0.067808
3,AII,2025-05-08,16.008226,1.005917,1.020118,1.039645,1.156213,1.414793,1.469231,1.156213,1.196450,1.049112,1.124852,1.139414,1.236239
4,AIRO,2025-06-13,24.000000,1.085417,0.955833,0.841667,0.836667,0.557083,0.354167,0.551250,0.383333,0.442500,0.360417,0.265417,0.329167


In [26]:
first_day_df[FUTURE_GROWTH].describe()

,future_growth_1_m,future_growth_2_m,future_growth_3_m,future_growth_4_m,future_growth_5_m,future_growth_6_m,future_growth_7_m,future_growth_8_m,future_growth_9_m,future_growth_10_m,future_growth_11_m,future_growth_12_m
count,132.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000
mean,1.119635,1.294861,1.252014,1.135939,1.084528,1.114789,1.146371,1.004473,1.158480,1.185497,1.068597,0.973337
std,1.254616,1.957999,1.912626,1.558464,1.468733,1.668273,2.278159,2.046907,2.999642,3.422082,2.161964,2.127421
min,0.072941,0.106197,0.090147,0.063927,0.032245,0.022075,0.013229,0.015130,0.014675,0.011947,0.008681,0.004833
25%,0.708835,0.584863,0.442477,0.392759,0.400704,0.308026,0.245548,0.200983,0.174587,0.174333,0.159666,0.159567
50%,0.935351,0.890411,0.827160,0.741678,0.667857,0.726000,0.662000,0.603508,0.580294,0.533333,0.492453,0.551321
75%,1.098038,1.152849,1.149994,1.203576,1.083509,1.056403,1.019297,1.036588,1.031006,1.034765,1.106045,1.057691
max,9.466019,16.066783,16.344464,11.386643,9.495146,9.894552,20.741478,21.370826,27.627298,34.687666,19.595802,22.044619


In [27]:
medians = first_day_df[FUTURE_GROWTH].median()
means = first_day_df[FUTURE_GROWTH].mean()
pd.DataFrame({'median': medians, 'mean': means})

,median,mean
future_growth_1_m,0.935351,1.119635
future_growth_2_m,0.890411,1.294861
future_growth_3_m,0.827160,1.252014
future_growth_4_m,0.741678,1.135939
future_growth_5_m,0.667857,1.084528
future_growth_6_m,0.726000,1.114789
future_growth_7_m,0.662000,1.146371
future_growth_8_m,0.603508,1.004473
future_growth_9_m,0.580294,1.158480
future_growth_10_m,0.533333,1.185497


In [28]:
print(f'Q3 answer: the highest median growth is {medians.max():.4f}, for {medians.idxmax()}')

Q3 answer: the highest median growth is 0.9354, for future_growth_1_m


# 4) Question 4: Simple RSI-Based Trading Strategy

Invest $1,000 every time a stock is oversold (`rsi < 30`) between 2000-01-01 and 2025-06-01, and use the 30-day forward return (`growth_future_30d`) as the result of each trade. The data is the course parquet file with precomputed indicators (about 130 MB).

In [29]:
# download the file only once (quiet=True: no progress bar in the notebook)
file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
if not os.path.exists("data.parquet"):
  gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=True)

df = pd.read_parquet("data.parquet", engine="pyarrow")
df.shape

(229932, 203)

In [30]:
df['Date'] = pd.to_datetime(df['Date'])

# oversold signal (RSI < 30) between 2000-01-01 and 2025-06-01
selected_df = df[(df.Date >= '2000-01-01') & (df.Date <= '2025-06-01') & (df.rsi < 30)]
print(f'Number of trades: {len(selected_df)}')

Number of trades: 5206


In [31]:
net_income = 1000 * (selected_df['growth_future_30d'] - 1).sum()

print(f"Average return per trade: {(selected_df['growth_future_30d'] - 1).mean():.2%}")
print(f"Win rate (trades with a positive return): {(selected_df['growth_future_30d'] > 1).mean():.2%}")
print(f'Q4 answer: net income = ${net_income:,.2f} = {net_income / 1000:.1f} thousand')

Average return per trade: 1.26%
Win rate (trades with a positive return): 55.13%
Q4 answer: net income = $65,805.59 = 65.8 thousand


In [32]:
# with last year's threshold (RSI < 25) there are far fewer trades
rsi_25_df = df[(df.Date >= '2000-01-01') & (df.Date <= '2025-06-01') & (df.rsi < 25)]
print(f'Number of trades with RSI < 25: {len(rsi_25_df)}')

Number of trades with RSI < 25: 1568


# 5) Question 5: How to make an IPO strategy more profitable

A few numbers from the 2025 IPOs above that support the ideas in the answer.

In [33]:
# how many IPOs are above their first-day close after 1, 3, 6 and 12 months
for m in [1, 3, 6, 12]:
  col = 'future_growth_'+str(m)+'_m'
  values = first_day_df[col].dropna()
  print(f'{m:>2} month(s): {(values > 1).mean():.0%} of the stocks are up; median growth {values.median():.2f}, mean {values.mean():.2f}, max {values.max():.1f}')

 1 month(s): 38% of the stocks are up; median growth 0.94, mean 1.12, max 9.5
 3 month(s): 34% of the stocks are up; median growth 0.83, mean 1.25, max 16.3
 6 month(s): 29% of the stocks are up; median growth 0.73, mean 1.11, max 9.9
12 month(s): 28% of the stocks are up; median growth 0.55, mean 0.97, max 22.0


In [34]:
# deal size = offer price x shares offered ($ millions)
first_day_df = pd.merge(first_day_df, ipos_2025_df[['Symbol', 'Offer Price', 'Shares (millions)']], left_on='Ticker', right_on='Symbol', how='left')
first_day_df['Offer Price'] = pd.to_numeric(first_day_df['Offer Price'].str.replace('$', ''), errors='coerce')
first_day_df['deal_size'] = first_day_df['Offer Price'] * first_day_df['Shares (millions)']

first_day_df['deal_size_group'] = pd.cut(first_day_df['deal_size'], bins=[0, 15, 50, 200, np.inf], labels=['< $15M', '$15-50M', '$50-200M', '> $200M'])
first_day_df['offer_price_group'] = pd.cut(first_day_df['Offer Price'], bins=[0, 5, 10, np.inf], labels=['<= $5', '$5-10', '> $10'])

In [35]:
# median growth by deal size
HORIZONS = ['future_growth_1_m', 'future_growth_3_m', 'future_growth_6_m', 'future_growth_12_m']
deal_size_stats = first_day_df.groupby('deal_size_group', observed=True)[HORIZONS].median()
deal_size_stats['number_of_ipos'] = first_day_df['deal_size_group'].value_counts()
deal_size_stats.round(2)

,future_growth_1_m,future_growth_3_m,future_growth_6_m,future_growth_12_m,number_of_ipos
deal_size_group,,,,,
< $15M,0.93,0.65,0.49,0.22,71
$15-50M,0.94,0.96,0.45,0.28,13
$50-200M,0.98,0.93,0.94,1.12,19
> $200M,0.93,0.83,0.89,0.78,29


In [36]:
# median growth by offer price
offer_price_stats = first_day_df.groupby('offer_price_group', observed=True)[HORIZONS].median()
offer_price_stats['number_of_ipos'] = first_day_df['offer_price_group'].value_counts()
offer_price_stats.round(2)

,future_growth_1_m,future_growth_3_m,future_growth_6_m,future_growth_12_m,number_of_ipos
offer_price_group,,,,,
<= $5,0.94,0.67,0.45,0.24,78
$5-10,0.99,0.93,0.68,0.38,10
> $10,0.92,0.85,0.90,0.84,44
